# Notebook 02 — Pixel Sampling & Time-Series Extraction

**Purpose:** turn the raw GEE assets into a single CSV of per-pixel monthly
time series, ready for downstream EDA and model training.

**Pipeline:**
1. Export a `burn_count + mask` GeoTIFF from GEE to Drive (one-off)
2. Sample pixel coordinates stratified by burn recurrence
3. Pull the monthly time series for every sampled pixel from GEE assets.
   Anomaly bands (z-scores, raw anomalies, dNBR) are computed **on the fly**
   from the composites and seasonal baselines, rather than read from a
   pre-exported anomaly collection. This avoids exporting ~180 GB of derived
   assets that can be reconstructed from the source data.

**Inputs:** GEE ImageCollections produced by Notebook 01
**Outputs:** `pixel_coords_10m.csv`, `full_pixels_10m.csv`


In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive", force_remount=False)


Mounted at /content/drive


In [2]:
# Load shared configuration.
# config.py must sit at /content/drive/MyDrive/thesis/config.py
import sys, importlib
sys.path.insert(0, "/content/drive/MyDrive/thesis")
import config
importlib.reload(config)

config.ensure_dirs()
config.set_seed()
config.print_summary()


Version:       _10m
GEE project:   bar-project-457815
Scale:         10 m   CRS: EPSG:32636
Period:        2017-2025  (107 months)
S2 collection: projects/bar-project-457815/assets/s2_monthly_image_collection_10m
Features:      27 model inputs (28 total available)
RPS weights:   {'recurrence': 0.25, 'severity': 0.25, 'persistence': 0.25, 'recovery': 0.25}
Seed:          42


In [3]:
import ee
import numpy as np
import pandas as pd
import os

try:
    ee.Initialize(project=config.GEE["project"])
except Exception:
    ee.Authenticate()
    ee.Initialize(project=config.GEE["project"])


## Step 1 - Export burn_count + mask as GeoTIFF

One-off export. Produces a two-band raster: total burn count per pixel across
the study period, plus the ecological mask. Step 2 reads this raster locally
to sample pixels by burn recurrence.

Run this cell, wait for the GEE task to complete (5-10 minutes), then continue.


In [ ]:
roi_fc = ee.FeatureCollection(config.GEE["roi"])
region = roi_fc.geometry()

ic_labels = (ee.ImageCollection(config.GEE["ic_labels"])
               .select("burned_this_month"))

landuse        = ee.FeatureCollection(config.GEE["landuse"])
human_infra_fc = ee.FeatureCollection(config.GEE["infra_mask"])

landuse_mask = (landuse
                .filter(ee.Filter.neq("FCODE", 501))
                .reduceToImage(["FCODE"], ee.Reducer.first())
                .gt(0))
human_infra_mask = (human_infra_fc
                    .reduceToImage(["Shape_Area"], ee.Reducer.count())
                    .gt(0))
final_mask = landuse_mask.And(human_infra_mask.Not())

burn_count = (ic_labels.sum()
              .rename("burn_count")
              .toInt()
              .clip(region))

mask_band = (final_mask.rename("ecological_mask")
             .toInt()
             .unmask(0)
             .clip(region))

combined = burn_count.addBands(mask_band)

task = ee.batch.Export.image.toDrive(
    image=combined,
    description=f"burn_count_export{config.VERSION}",
    folder="thesis_data_raw",  # GEE folder param does not support nested paths; flat folder name only
    fileNamePrefix=f"burn_count_with_mask{config.VERSION}",
    region=region,
    scale=config.SCALE,
    crs=config.CRS,
    maxPixels=config.MAX_PIXELS,
    fileFormat="GeoTIFF",
)
task.start()

print(f"Export task started: burn_count_export{config.VERSION}")
print(f"Output file: {config.FILES['burn_count_tif']}")
print("Wait 5-10 minutes for completion.")
print("Check status: https://code.earthengine.google.com/tasks")


Export task started: burn_count_export_10m
Output file: /content/drive/MyDrive/thesis/data/raw/burn_count_with_mask_10m.tif
Wait 5-10 minutes for completion.
Check status: https://code.earthengine.google.com/tasks


## Step 2 - Sample pixel coordinates stratified by burn recurrence

In [ ]:
!pip install -q pyproj rasterio


In [ ]:
import rasterio
from rasterio.transform import xy
import pyproj

TIF_PATH    = config.FILES["burn_count_tif"]
COORDS_PATH = config.FILES["pixel_coords"]

with rasterio.open(TIF_PATH) as src:
    burn_count = src.read(1)
    eco_mask   = src.read(2)
    transform  = src.transform
    src_crs    = src.crs
    print(f"Raster shape: {burn_count.shape}")

mask_low    = (burn_count >= 1) & (burn_count <= 2)
mask_medium = (burn_count >= 3) & (burn_count <= 4)
mask_high   = (burn_count >= 5)
mask_bg     = (burn_count == 0) & (eco_mask == 1)

print("\nPixel availability:")
print(f"  burned_low    (1-2 fires):  {int(mask_low.sum()):,}")
print(f"  burned_medium (3-4 fires):  {int(mask_medium.sum()):,}")
print(f"  burned_high   (5+ fires):   {int(mask_high.sum()):,}")
print(f"  never_burned  (0 fires):    {int(mask_bg.sum()):,}")


Raster shape: (7792, 4874)

Pixel availability:
  burned_low    (1-2 fires):  254,698
  burned_medium (3-4 fires):  3,440
  burned_high   (5+ fires):   323
  never_burned  (0 fires):    16,209,218


In [ ]:
def sample_mask(mask, n_want, label):
    rows, cols = np.where(mask)
    n_avail = len(rows)
    n_take  = min(n_want, n_avail)
    print(f"  {label}: take {n_take:,} of {n_avail:,}")
    if n_avail == 0:
        return []
    idx = np.random.choice(n_avail, size=n_take, replace=False)
    return list(zip(rows[idx], cols[idx]))


print("Sampling pixels by stratum:")
pix_low    = sample_mask(mask_low,    config.SAMPLING["n_burned_low"],    "burned_low")
pix_medium = sample_mask(mask_medium, config.SAMPLING["n_burned_medium"], "burned_medium")
pix_high   = sample_mask(mask_high,   config.SAMPLING["n_burned_high"],   "burned_high")
pix_bg     = sample_mask(mask_bg,     config.SAMPLING["n_never_burned"],  "never_burned")


Sampling pixels by stratum:
  burned_low: take 4,500 of 254,698
  burned_medium: take 800 of 3,440
  burned_high: take 200 of 323
  never_burned: take 4,500 of 16,209,218


In [ ]:
transformer = pyproj.Transformer.from_crs(src_crs, "EPSG:4326", always_xy=True)


def to_lonlat(pixel_list, ptype):
    out = []
    for r, c in pixel_list:
        x, y     = xy(transform, r, c, offset="center")
        lon, lat = transformer.transform(x, y)
        out.append({"lon": lon, "lat": lat, "pixel_type": ptype})
    return out


records = []
records += to_lonlat(pix_low,    "burned_low")
records += to_lonlat(pix_medium, "burned_medium")
records += to_lonlat(pix_high,   "burned_high")
records += to_lonlat(pix_bg,     "never_burned")

for i, p in enumerate(records):
    p["pixel_id"] = i

df_pixels = pd.DataFrame(records)[["pixel_id", "lon", "lat", "pixel_type"]]
df_pixels.to_csv(COORDS_PATH, index=False)

print(f"\nTotal pixels: {len(df_pixels)}")
print(df_pixels["pixel_type"].value_counts().to_string())
print(f"\nSaved: {COORDS_PATH}")



Total pixels: 10000
pixel_type
burned_low       4500
never_burned     4500
burned_medium     800
burned_high       200

Saved: /content/drive/MyDrive/thesis/data/raw/pixel_coords_10m.csv


## Step 3 - Extract monthly time series for every sampled pixel
todo- I ran steps 1+2, need to run step 3.


For each (year, month), loads the monthly composite and label image, and
**computes the anomaly bands on the fly** from the composite plus the
seasonal baseline (mean, median, std). Samples all bands at the pixel
coordinates. Result: one row per (pixel, month) with all features and the
burn label.

Slow step: expect a few hours of GEE queries.


In [4]:
SAVE_PATH = config.FILES["raw_timeseries"]

ic_s2       = ee.ImageCollection(config.GEE["ic_s2"])
ic_labels   = (ee.ImageCollection(config.GEE["ic_labels"])
                 .select("burned_this_month"))
ic_b_mean   = ee.ImageCollection(config.GEE["ic_baseline_mean"])
ic_b_median = ee.ImageCollection(config.GEE["ic_baseline_median"])
ic_b_std    = ee.ImageCollection(config.GEE["ic_baseline_std"])

S2_BANDS      = config.RAW_BANDS + config.INDICES + config.QUALITY_BANDS
ANOMALY_BANDS = config.ANOMALY_BANDS
ANOMALY_SRC   = config.ANOMALY_SOURCE_INDICES
ALL_FEATURES  = config.ALL_FEATURES

MONTHS = config.get_month_list()

df_coords = pd.read_csv(config.FILES["pixel_coords"])
pixels    = df_coords.to_dict("records")
print(f"Pixels:  {len(pixels):,}")
print(f"Months:  {len(MONTHS)}")


Pixels:  10,000
Months:  107


In [5]:
def get_baseline(ic, month, bands):
    """Return the baseline image for a given calendar month."""
    return ee.Image(
        ic.filter(ee.Filter.eq("month", month)).first()
    ).select(bands)


def compute_anomaly_image(observed_s2, month):
    """Compute the 15 anomaly bands (7 z-scores + 7 raw anomalies + dNBR)
    on the fly from the observed composite and the seasonal baseline.

    Replicates the formulas used in Notebook 01's Step 6 export so that the
    pipeline downstream is identical to working with a pre-exported anomaly
    collection. No anomaly asset is required.
    """
    observed = observed_s2.select(ANOMALY_SRC).unmask(-9999)

    b_mean   = get_baseline(ic_b_mean,   month, ANOMALY_SRC)
    b_median = get_baseline(ic_b_median, month, ANOMALY_SRC)
    b_std    = get_baseline(ic_b_std,    month, ANOMALY_SRC)

    eps = ee.Image.constant(1e-6)

    z_score = (observed
               .subtract(b_mean)
               .divide(b_std.add(eps))
               .rename([b + "_zscore" for b in ANOMALY_SRC]))

    raw_anomaly = (observed
                   .subtract(b_median)
                   .rename([b + "_anomaly" for b in ANOMALY_SRC]))

    dnbr = (b_median.select("NBR")
            .subtract(observed.select("NBR"))
            .rename("NBR_dnbr"))

    return z_score.addBands(raw_anomaly).addBands(dnbr)


In [6]:
BATCH_SIZE = 500
n_batches  = (len(pixels) + BATCH_SIZE - 1) // BATCH_SIZE

print(f"Extracting {len(MONTHS)} months x {len(pixels)} pixels")
print(f"  Batched into {n_batches} groups of {BATCH_SIZE}")
print(f"  Estimated time: 3-5 hours\n")

all_rows = []

for t, (year, month) in enumerate(MONTHS):

    if t % 5 == 0:
        print(f"  [{t+1}/{len(MONTHS)}] {year}-{month:02d} "
              f"- rows so far: {len(all_rows):,}")

    try:
        has_s2 = (ic_s2.filter(ee.Filter.eq("year", year))
                       .filter(ee.Filter.eq("month", month))
                       .size().gt(0))
        has_lbl = (ic_labels.filter(ee.Filter.eq("year", year))
                            .filter(ee.Filter.eq("month", month))
                            .size().gt(0))

        # If the composite for this month is missing, fall back to placeholder
        # values; downstream NaN handling will catch them.
        s2_img = ee.Image(ee.Algorithms.If(
            has_s2,
            (ic_s2.filter(ee.Filter.eq("year", year))
                  .filter(ee.Filter.eq("month", month))
                  .first().select(S2_BANDS).unmask(-9999)),
            ee.Image.constant([-9999]*len(S2_BANDS))
              .rename(S2_BANDS).toFloat()
        ))

        # Anomaly is computed on the fly from the composite + baselines
        anom_img = ee.Image(ee.Algorithms.If(
            has_s2,
            compute_anomaly_image(
                ic_s2.filter(ee.Filter.eq("year", year))
                     .filter(ee.Filter.eq("month", month))
                     .first(),
                month
            ).unmask(-9999),
            ee.Image.constant([-9999]*len(ANOMALY_BANDS))
              .rename(ANOMALY_BANDS).toFloat()
        ))

        lbl_img = ee.Image(ee.Algorithms.If(
            has_lbl,
            (ic_labels.filter(ee.Filter.eq("year", year))
                      .filter(ee.Filter.eq("month", month))
                      .first().unmask(0)),
            ee.Image.constant(0).rename(["burned_this_month"]).toFloat()
        ))

        month_img = s2_img.addBands(anom_img).addBands(lbl_img)

        for batch_idx in range(n_batches):
            lo = batch_idx * BATCH_SIZE
            hi = min(lo + BATCH_SIZE, len(pixels))
            batch_pixels = pixels[lo:hi]

            batch_fc = ee.FeatureCollection([
                ee.Feature(
                    ee.Geometry.Point([p["lon"], p["lat"]]),
                    {"pixel_id":   int(p["pixel_id"]),
                     "pixel_type": p["pixel_type"]}
                )
                for p in batch_pixels
            ])

            sampled = month_img.reduceRegions(
                collection=batch_fc,
                reducer=ee.Reducer.first(),
                scale=config.SCALE,
                crs=config.CRS,
            ).getInfo()["features"]

            for feat in sampled:
                props = feat["properties"]
                row = {
                    "pixel_id":   props.get("pixel_id"),
                    "pixel_type": props.get("pixel_type"),
                    "year":       year,
                    "month":      month,
                    "burned":     int(props.get("burned_this_month", 0) or 0),
                }
                for b in ALL_FEATURES:
                    row[b] = props.get(b, -9999)
                all_rows.append(row)

    except Exception as e:
        print(f"  WARNING: {year}-{month:02d}: {e}")
        continue

print(f"\nDone. Total rows: {len(all_rows):,}")


Extracting 107 months x 10000 pixels
  Batched into 20 groups of 500
  Estimated time: 3-5 hours

  [1/107] 2017-02 - rows so far: 0
  [6/107] 2017-07 - rows so far: 50,000
  [11/107] 2017-12 - rows so far: 100,000
  [16/107] 2018-05 - rows so far: 150,000
  [21/107] 2018-10 - rows so far: 200,000


  [26/107] 2019-03 - rows so far: 250,000
  [31/107] 2019-08 - rows so far: 300,000
  [36/107] 2020-01 - rows so far: 350,000


  [41/107] 2020-06 - rows so far: 400,000
  [46/107] 2020-11 - rows so far: 450,000
  [51/107] 2021-04 - rows so far: 500,000
  [56/107] 2021-09 - rows so far: 550,000
  [61/107] 2022-02 - rows so far: 600,000


  [66/107] 2022-07 - rows so far: 650,000
  [71/107] 2022-12 - rows so far: 700,000
  [76/107] 2023-05 - rows so far: 750,000
  [81/107] 2023-10 - rows so far: 800,000
  [86/107] 2024-03 - rows so far: 850,000
  [91/107] 2024-08 - rows so far: 900,000


  [96/107] 2025-01 - rows so far: 950,000
  [101/107] 2025-06 - rows so far: 1,000,000
  [106/107] 2025-11 - rows so far: 1,050,000

Done. Total rows: 1,070,000


In [7]:
df = pd.DataFrame(all_rows)
df.replace(-9999, np.nan, inplace=True)

df_clean = df[df["pct_valid"] >= config.SAMPLING["quality_threshold"]].copy()

print(f"Rows before quality filter: {len(df):,}")
print(f"Rows after  quality filter: {len(df_clean):,}")
print(f"Burned rows: {int(df_clean['burned'].sum()):,} "
      f"({100*df_clean['burned'].mean():.2f}%)")
print()
print("Pixel-type distribution (after filter):")
print(df_clean.groupby("pixel_type")["pixel_id"].nunique().to_string())

df_clean.to_csv(SAVE_PATH, index=False)
print(f"\nSaved: {SAVE_PATH}")
print(f"File size: {os.path.getsize(SAVE_PATH)/(1024*1024):.1f} MB")


Rows before quality filter: 1,070,000
Rows after  quality filter: 992,679
Burned rows: 8,094 (0.82%)

Pixel-type distribution (after filter):
pixel_type
burned_high       200
burned_low       4194
burned_medium     773
never_burned     4500

Saved: /content/drive/MyDrive/thesis/data/raw/full_pixels_10m.csv
File size: 514.3 MB
